# Duas etapas — ápices (DENTEX) e lesões em recortes ampliados (Hanói) — ML Odonto

1. Treina um **localizador de ápices radiculares** com os contornos de dentes do DENTEX (634 panorâmicas, uso não comercial).
2. Usa o localizador para achar os ápices nas 3.924 panorâmicas de Hanói (Mendeley, CC BY 4.0).
3. Recorta 256 × 256 px em volta de cada ápice, na resolução original (~2800 px de largura).
4. Treina uma **U-Net de lesão** nesses recortes (perda de Tversky).

Só dados públicos vêm para o Colab; a avaliação nas radiografias de pacientes é feita no computador local.
Tudo fica em `MyDrive/ML_Odonto_DuasEtapas`; se a sessão cair, rode tudo de novo: as fases prontas são puladas.

Tempo esperado (GPU T4): ~15 min de downloads, ~25 min de localizador, ~25 min de preparo dos recortes, ~35 min de segmentador.

### 1. GPU

In [ ]:
!nvidia-smi -L
import os; print('CPUs:', os.cpu_count())

### 2. Google Drive (onde ficam modelos e resultados)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAIDA = '/content/drive/MyDrive/ML_Odonto_DuasEtapas'
!mkdir -p "{SAIDA}" /content/scripts

### 3. Código do projeto (gerado a partir de `scripts/`)

In [ ]:
%%writefile /content/scripts/comum.py
"""Peças compartilhadas: carregamento, normalização, atributos por pixel e métricas."""
import xml.etree.ElementTree as ET
from pathlib import Path

import cv2
import numpy as np
from scipy import ndimage as ndi
from skimage.feature import hessian_matrix, hessian_matrix_eigvals

RAIZ = Path(__file__).resolve().parent.parent
NOSSAS = RAIZ / "Treinamento" / "extraido"
HANOI = RAIZ / "Externos" / "Hanoi"
MODELOS = RAIZ / "Modelos"
RESULTADOS = RAIZ / "Resultados"

# Todas as panorâmicas são levadas a esta largura antes de tudo: as originais têm
# ~2400-2900 px e as lesões ~60-100 px, que viram ~30-50 px aqui.
LARGURA = 1400


def ler_cinza(caminho):
    # imdecode lida com acentos no caminho (cv2.imread não, no Windows)
    dados = np.fromfile(str(caminho), np.uint8)
    img = cv2.imdecode(dados, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"não consegui ler {caminho}")
    return img


def normalizar(img, largura=None):
    """Redimensiona para `largura` (padrão LARGURA) e equaliza o contraste (CLAHE) para aproximar aparelhos diferentes."""
    largura = largura or LARGURA
    escala = largura / img.shape[1]
    interp = cv2.INTER_AREA if escala <= 1 else cv2.INTER_CUBIC
    img = cv2.resize(img, (largura, round(img.shape[0] * escala)), interpolation=interp)
    img = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)
    return img, escala


def redimensionar_mascara(m, forma):
    return cv2.resize(m, (forma[1], forma[0]), interpolation=cv2.INTER_NEAREST) > 0


NOMES_ATRIBUTOS = []
VERSAO_ATRIBUTOS = 2  # muda quando atributos() muda, para invalidar amostras em cache


def deslocar(A, dy, dx):
    """Valor de A no ponto (y+dy, x+dx), repetindo a borda fora da imagem."""
    p = max(abs(dy), abs(dx))
    Ap = np.pad(A, p, mode="edge")
    h, w = A.shape
    return Ap[p + dy:p + dy + h, p + dx:p + dx + w]


def atributos(img):
    """Pilha (H, W, F) de atributos por pixel, no estilo ilastik/Weka.

    A lesão periapical é uma área mais escura que o osso ao redor, perto da ponta
    da raiz: por isso brilho em várias escalas, contraste com a vizinhança (DoG),
    bordas, forma local (Hessiana), textura e posição na imagem.
    """
    I = img.astype(np.float32) / 255.0
    h, w = I.shape
    camadas, nomes = [I], ["brilho"]
    G = {s: cv2.GaussianBlur(I, (0, 0), s) for s in (1, 2, 4, 8, 16, 32)}
    for s in (1, 2, 4, 8, 16):
        camadas.append(G[s]); nomes.append(f"gauss_{s}")
    for a, b in ((1, 4), (2, 8), (4, 16), (8, 32)):
        camadas.append(G[a] - G[b]); nomes.append(f"dog_{a}_{b}")
    for s in (1, 2, 4, 8):
        gx = cv2.Sobel(G[s], cv2.CV_32F, 1, 0)
        gy = cv2.Sobel(G[s], cv2.CV_32F, 0, 1)
        camadas.append(np.sqrt(gx * gx + gy * gy)); nomes.append(f"gradiente_{s}")
    for s in (2, 4, 8):
        H = hessian_matrix(I, sigma=s, order="rc", use_gaussian_derivatives=False)
        l1, l2 = hessian_matrix_eigvals(H)
        camadas += [l1.astype(np.float32), l2.astype(np.float32)]
        nomes += [f"hessiana_{s}_l1", f"hessiana_{s}_l2"]
    for s in (4, 8):
        m2 = cv2.GaussianBlur(I * I, (0, 0), s)
        camadas.append(np.sqrt(np.maximum(m2 - G[s] ** 2, 0))); nomes.append(f"desvio_{s}")
    # Contexto: a lesão fica na ponta de uma raiz, então o brilho alguns pixels
    # acima/abaixo/ao lado (raiz clara? osso? ar?) diz muito sobre o pixel central.
    for s in (4, 8):
        for dy, dx in ((-40, 0), (-20, 0), (20, 0), (40, 0), (0, -25), (0, 25)):
            camadas.append(deslocar(G[s], dy, dx) - G[s]); nomes.append(f"contexto_{s}_{dy}_{dx}")
    camadas.append(G[32] - cv2.GaussianBlur(I, (0, 0), 64)); nomes.append("dog_32_64")
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    camadas += [xx / w, yy / h, np.abs(xx / w - 0.5)]
    nomes += ["pos_x", "pos_y", "dist_centro_x"]
    if not NOMES_ATRIBUTOS:
        NOMES_ATRIBUTOS.extend(nomes)
    return np.stack(camadas, axis=-1)


# ---------------------------------------------------------------- conjuntos

def nossas_imagens():
    """[(id, img_normalizada, mascara_bool)] das 20 panorâmicas marcadas do PDF."""
    saida = []
    for f in sorted((NOSSAS / "imagens").glob("*.png")):
        img, _ = normalizar(ler_cinza(f))
        m = redimensionar_mascara(ler_cinza(NOSSAS / "mascaras" / f.name), img.shape)
        saida.append((f.stem, img, m))
    return saida


def caixas_hanoi(nome):
    r = ET.parse(HANOI / "anotacoes" / "Image Annots" / f"{nome}.xml").getroot()
    caixas = []
    for o in r.iter("object"):
        b = o.find("bndbox")
        caixas.append(tuple(float(b.findtext(t)) for t in ("xmin", "ymin", "xmax", "ymax")))
    return caixas


def imagem_hanoi(nome, largura=None):
    """Imagem normalizada, máscara (elipse dentro de cada caixa) e zona incerta.

    A caixa é justa em torno da lesão, então a elipse inscrita é uma boa
    aproximação. A faixa entre a elipse e a caixa ampliada é marcada como
    incerta e não entra no treino nem como lesão nem como fundo.
    """
    img, esc = normalizar(ler_cinza(HANOI / "imagens" / "Original JPG Images" / f"{nome}.jpg"), largura)
    mascara = np.zeros(img.shape, np.uint8)
    incerta = np.zeros(img.shape, np.uint8)
    for x0, y0, x1, y1 in caixas_hanoi(nome):
        cx, cy = (x0 + x1) / 2 * esc, (y0 + y1) / 2 * esc
        rx, ry = (x1 - x0) / 2 * esc, (y1 - y0) / 2 * esc
        cv2.ellipse(incerta, (round(cx), round(cy)), (round(rx * 1.5), round(ry * 1.5)), 0, 0, 360, 1, -1)
        cv2.ellipse(mascara, (round(cx), round(cy)), (round(rx * 0.85), round(ry * 0.85)), 0, 0, 360, 1, -1)
    return img, mascara > 0, (incerta > 0) & (mascara == 0)


def nomes_hanoi():
    return sorted(p.stem for p in (HANOI / "imagens" / "Original JPG Images").glob("*.jpg"))


# ---------------------------------------------------------------- amostragem

def amostrar(F, mascara, incerta, n_pos, n_neg, rng):
    """Sorteia pixels de lesão e de fundo. Metade do fundo vem da faixa dos dentes,
    onde estão as confusões difíceis (raízes, espaços entre dentes)."""
    h, w = mascara.shape
    pos = np.flatnonzero(mascara)
    neg_ok = ~mascara & ~incerta
    faixa = np.zeros_like(neg_ok)
    faixa[int(0.25 * h):int(0.85 * h), int(0.15 * w):int(0.85 * w)] = True
    neg_todos = np.flatnonzero(neg_ok)
    neg_faixa = np.flatnonzero(neg_ok & faixa)
    escolha = lambda arr, n: rng.choice(arr, min(n, len(arr)), replace=False) if len(arr) else arr
    idx_pos = escolha(pos, n_pos)
    idx_neg = np.concatenate([escolha(neg_todos, n_neg // 2), escolha(neg_faixa, n_neg - n_neg // 2)])
    Fp = F.reshape(-1, F.shape[-1])
    X = np.concatenate([Fp[idx_pos], Fp[idx_neg]])
    y = np.concatenate([np.ones(len(idx_pos), np.uint8), np.zeros(len(idx_neg), np.uint8)])
    return X, y


# ---------------------------------------------------------------- predição

def prever_probabilidade(modelo, img, lote=200_000, devolver_atributos=False):
    F = atributos(img)
    Fp = F.reshape(-1, F.shape[-1])
    p = np.concatenate([modelo.predict_proba(Fp[i:i + lote])[:, 1] for i in range(0, len(Fp), lote)])
    p = p.reshape(img.shape).astype(np.float32)
    return (p, F) if devolver_atributos else p


def segmentar(prob, limiar, area_min):
    """Probabilidade -> máscara: suaviza, corta no limiar e descarta manchas pequenas."""
    p = cv2.GaussianBlur(prob, (0, 0), 2)
    m = (p >= limiar).astype(np.uint8)
    n, rot, st, _ = cv2.connectedComponentsWithStats(m)
    manter = np.zeros(n, bool)
    manter[1:] = st[1:, cv2.CC_STAT_AREA] >= area_min
    return manter[rot]


# ---------------------------------------------------------------- métricas

FRACAO_ACERTO = 0.2


def avaliar(pred, verdade):
    """Por imagem: acertos de pixel e de lesão.

    Uma mancha prevista só é acerto se pelo menos FRACAO_ACERTO dela cair sobre
    lesão de verdade; assim uma mancha enorme que só encosta na lesão não conta.
    Uma lesão verdadeira é achada se alguma mancha correta a toca.
    """
    inter = (pred & verdade).sum()
    nv, rv = cv2.connectedComponents(verdade.astype(np.uint8))
    np_, rp = cv2.connectedComponents(pred.astype(np.uint8))
    area = np.bincount(rp.ravel(), minlength=np_)
    sobre = np.bincount(rp[verdade], minlength=np_)
    corretas = np.flatnonzero(sobre >= FRACAO_ACERTO * np.maximum(area, 1))
    corretas = corretas[corretas > 0]
    achadas = len(set(np.unique(rv[np.isin(rp, corretas)])) - {0})
    return dict(inter=int(inter), soma=int(pred.sum() + verdade.sum()),
                lesoes=nv - 1, lesoes_achadas=achadas,
                previstas=np_ - 1, previstas_corretas=len(corretas))


def resumir(linhas):
    s = {k: sum(l[k] for l in linhas) for k in linhas[0]}
    return dict(
        dice=2 * s["inter"] / max(s["soma"], 1),
        sensibilidade=s["lesoes_achadas"] / max(s["lesoes"], 1),
        precisao=s["previstas_corretas"] / max(s["previstas"], 1),
        falsos_por_imagem=(s["previstas"] - s["previstas_corretas"]) / len(linhas),
    )

In [ ]:
%%writefile /content/scripts/unet.py
"""U-Net com codificador ResNet34 (torchvision), usada no Colab para treinar e aqui para prever.

Só depende de torch e torchvision, para o mesmo arquivo rodar nos dois lugares.
"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

MEDIA = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
DESVIO = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class Bloco(nn.Sequential):
    def __init__(self, entrada, saida):
        super().__init__(
            nn.Conv2d(entrada, saida, 3, padding=1, bias=False), nn.BatchNorm2d(saida), nn.ReLU(inplace=True),
            nn.Conv2d(saida, saida, 3, padding=1, bias=False), nn.BatchNorm2d(saida), nn.ReLU(inplace=True))


class UNetResNet34(nn.Module):
    def __init__(self, pre_treinada=False):
        super().__init__()
        r = torchvision.models.resnet34(weights="IMAGENET1K_V1" if pre_treinada else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)                   # 1/2, 64
        self.pool = r.maxpool
        self.c1, self.c2, self.c3, self.c4 = r.layer1, r.layer2, r.layer3, r.layer4  # 1/4 .. 1/32
        self.d4 = Bloco(512 + 256, 256)
        self.d3 = Bloco(256 + 128, 128)
        self.d2 = Bloco(128 + 64, 64)
        self.d1 = Bloco(64 + 64, 64)
        self.d0 = Bloco(64, 32)
        self.saida = nn.Conv2d(32, 1, 1)
        self.register_buffer("media", MEDIA.clone(), persistent=False)
        self.register_buffer("desvio", DESVIO.clone(), persistent=False)

    def forward(self, x):
        """x: (N, 1, H, W) em [0, 1], H e W múltiplos de 32. Devolve logits (N, 1, H, W)."""
        x = (x.repeat(1, 3, 1, 1) - self.media) / self.desvio
        s1 = self.stem(x)
        s2 = self.c1(self.pool(s1))
        s3 = self.c2(s2)
        s4 = self.c3(s3)
        s5 = self.c4(s4)
        sobe = lambda t: F.interpolate(t, scale_factor=2, mode="bilinear", align_corners=False)
        d = self.d4(torch.cat([sobe(s5), s4], 1))
        d = self.d3(torch.cat([sobe(d), s3], 1))
        d = self.d2(torch.cat([sobe(d), s2], 1))
        d = self.d1(torch.cat([sobe(d), s1], 1))
        return self.saida(self.d0(sobe(d)))


def carregar(caminho, dispositivo="cpu"):
    ck = torch.load(caminho, map_location=dispositivo, weights_only=False)
    m = UNetResNet34()
    m.load_state_dict(ck["modelo"])
    return m.to(dispositivo).eval(), ck.get("meta", {})


@torch.no_grad()
def prever(modelo, img, espelhar=True):
    """Mapa de probabilidade (H, W) para uma imagem uint8 já normalizada (comum.normalizar).

    Com espelhar=True faz a média com a previsão da imagem espelhada (a panorâmica
    é quase simétrica), o que costuma estabilizar o resultado.
    """
    h, w = img.shape
    H, W = -(-h // 32) * 32, -(-w // 32) * 32
    x = np.pad(img, ((0, H - h), (0, W - w)), mode="reflect").astype(np.float32) / 255.0
    disp = next(modelo.parameters()).device
    t = torch.from_numpy(x)[None, None].to(disp)
    p = torch.sigmoid(modelo(t))
    if espelhar:
        p = (p + torch.sigmoid(modelo(t.flip(-1))).flip(-1)) / 2
    return p[0, 0, :h, :w].float().cpu().numpy()

In [ ]:
%%writefile /content/scripts/treino_unet.py
"""Treina a U-Net com as panorâmicas de Hanói (feito para o Google Colab com GPU).

Usa o mesmo pré-processamento do Random Forest (comum.normalizar) e as mesmas
máscaras: elipse dentro de cada caixa = lesão, anel em volta = ignorado na perda.
Salva checkpoints em --saida a cada validação; com --continuar retoma de lá.

Versão 2 (lesões pequenas): --largura 2000 treina em resolução maior e --perda tversky
penaliza mais deixar lesão passar do que marcar a mais.

Uso: python treino_unet.py --saida PASTA [--iteracoes 6000] [--lote 8] [--largura 1400]
                           [--perda dice|tversky] [--continuar]
"""
import argparse
import math
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from joblib import Parallel, delayed
from torch.utils.data import DataLoader, Dataset

import comum as c
from unet import UNetResNet34, prever

RECORTE = (512, 768)  # altura, largura dos recortes de treino
IGNORAR = 2


def preparar_um(nome, pasta, largura):
    destino = pasta / f"{nome}_img.png"
    if destino.exists():
        return
    img, m, inc = c.imagem_hanoi(nome, largura)
    rot = np.zeros(img.shape, np.uint8)
    rot[inc] = IGNORAR
    rot[m] = 1
    cv2.imwrite(str(pasta / f"{nome}_rot.png"), rot)
    cv2.imwrite(str(destino), img)


def preparar(nomes, pasta, workers, largura):
    pasta.mkdir(parents=True, exist_ok=True)
    t = time.time()
    Parallel(n_jobs=workers)(delayed(preparar_um)(nm, pasta, largura) for nm in nomes)
    print(f"Pré-processamento: {len(nomes)} imagens em {time.time() - t:.0f}s", flush=True)


class Recortes(Dataset):
    """Recortes aleatórios; metade centrada numa lesão, para ela não ser rara demais."""

    def __init__(self, nomes, pasta, n_amostras, semente=0):
        self.nomes, self.pasta, self.n = nomes, pasta, n_amostras
        self.semente = semente

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        rng = np.random.default_rng(self.semente * 1_000_003 + i)
        nm = self.nomes[rng.integers(len(self.nomes))]
        img = cv2.imread(str(self.pasta / f"{nm}_img.png"), cv2.IMREAD_GRAYSCALE)
        rot = cv2.imread(str(self.pasta / f"{nm}_rot.png"), cv2.IMREAD_GRAYSCALE)
        ch, cw = RECORTE
        if img.shape[0] < ch or img.shape[1] < cw:
            py, px = max(0, ch - img.shape[0]), max(0, cw - img.shape[1])
            img = np.pad(img, ((0, py), (0, px)), mode="reflect")
            rot = np.pad(rot, ((0, py), (0, px)), constant_values=IGNORAR)
        h, w = img.shape
        ys, xs = np.nonzero(rot == 1)
        if len(ys) and rng.random() < 0.5:
            k = rng.integers(len(ys))
            y0 = np.clip(ys[k] - rng.integers(ch // 4, 3 * ch // 4), 0, h - ch)
            x0 = np.clip(xs[k] - rng.integers(cw // 4, 3 * cw // 4), 0, w - cw)
        else:
            y0, x0 = rng.integers(0, h - ch + 1), rng.integers(0, w - cw + 1)
        img = img[y0:y0 + ch, x0:x0 + cw].astype(np.float32) / 255.0
        rot = rot[y0:y0 + ch, x0:x0 + cw]
        if rng.random() < 0.5:
            img, rot = img[:, ::-1], rot[:, ::-1]
        img = np.clip(img ** rng.uniform(0.7, 1.4) * rng.uniform(0.85, 1.15) + rng.uniform(-0.08, 0.08), 0, 1)
        return torch.from_numpy(img.copy())[None], torch.from_numpy(rot.copy()).long()


TVERSKY_FP, TVERSKY_FN = 0.3, 0.7  # peso de marcar a mais × peso de deixar lesão passar


def perda(logits, rot, tipo="dice"):
    """BCE + Dice (ou Tversky), só nos pixels que não estão na zona ignorada."""
    logits = logits[:, 0]
    valido = (rot != IGNORAR).float()
    alvo = (rot == 1).float()
    bce = (F.binary_cross_entropy_with_logits(logits, alvo, reduction="none") * valido).sum() / valido.sum()
    p = torch.sigmoid(logits) * valido
    vp = (p * alvo).sum()
    if tipo == "tversky":
        fp, fn = (p * (1 - alvo)).sum(), ((1 - p) * alvo).sum()
        return bce + 1 - (vp + 1) / (vp + TVERSKY_FP * fp + TVERSKY_FN * fn + 1)
    dice = 1 - (2 * vp + 1) / (p.sum() + alvo.sum() + 1)
    return bce + dice


def validar(modelo, nomes, pasta):
    modelo.eval()
    inter = soma = 0
    for nm in nomes:
        img = cv2.imread(str(pasta / f"{nm}_img.png"), cv2.IMREAD_GRAYSCALE)
        rot = cv2.imread(str(pasta / f"{nm}_rot.png"), cv2.IMREAD_GRAYSCALE)
        pred = prever(modelo, img, espelhar=False) >= 0.5
        valido = rot != IGNORAR
        inter += (pred & (rot == 1)).sum()
        soma += (pred & valido).sum() + (rot == 1).sum()
    modelo.train()
    return 2 * inter / max(soma, 1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--saida", required=True)
    ap.add_argument("--iteracoes", type=int, default=6000)
    ap.add_argument("--lote", type=int, default=8)
    ap.add_argument("--validar-cada", type=int, default=500)
    ap.add_argument("--n-val", type=int, default=200)
    ap.add_argument("--limite-imagens", type=int, default=0, help="só para testes rápidos")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--continuar", action="store_true")
    ap.add_argument("--largura", type=int, default=c.LARGURA, help="largura (px) em que as panorâmicas são treinadas")
    ap.add_argument("--perda", choices=("dice", "tversky"), default="dice")
    ap.add_argument("--sem-pre-treino", action="store_true", help="não baixa pesos ImageNet (testes)")
    args = ap.parse_args()

    disp = "cuda" if torch.cuda.is_available() else "cpu"
    saida = Path(args.saida)
    saida.mkdir(parents=True, exist_ok=True)
    nomes = c.nomes_hanoi()
    np.random.default_rng(0).shuffle(nomes)
    if args.limite_imagens:
        nomes = nomes[:args.limite_imagens]
    n_val = min(args.n_val, len(nomes) // 5)
    val, treino = nomes[:n_val], nomes[n_val:]
    pasta = c.HANOI / f"preparadas_{args.largura}"
    preparar(nomes, pasta, max(args.workers, 1), args.largura)
    print(f"Treino: {len(treino)} imagens | validação: {len(val)} | dispositivo: {disp}", flush=True)

    modelo = UNetResNet34(pre_treinada=not args.sem_pre_treino).to(disp)
    opt = torch.optim.AdamW(modelo.parameters(), lr=3e-4, weight_decay=1e-4)
    agenda = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda it: min(1, (it + 1) / 200) * 0.5 * (1 + math.cos(math.pi * it / args.iteracoes)))
    escalador = torch.amp.GradScaler(enabled=disp == "cuda")
    inicio, melhor = 0, -1.0
    ck_path = saida / "checkpoint.pt"
    if args.continuar and ck_path.exists():
        ck = torch.load(ck_path, map_location=disp, weights_only=False)
        modelo.load_state_dict(ck["modelo"]); opt.load_state_dict(ck["opt"]); agenda.load_state_dict(ck["agenda"])
        inicio, melhor = ck["iteracao"], ck["melhor"]
        print(f"Retomando da iteração {inicio} (melhor Dice de validação {melhor:.3f})", flush=True)

    dados = DataLoader(Recortes(treino, pasta, (args.iteracoes - inicio) * args.lote, semente=inicio),
                       batch_size=args.lote, num_workers=args.workers, pin_memory=disp == "cuda")
    modelo.train()
    t, acum = time.time(), 0.0
    for it, (x, y) in enumerate(dados, start=inicio + 1):
        x, y = x.to(disp, non_blocking=True), y.to(disp, non_blocking=True)
        with torch.autocast(device_type=disp, dtype=torch.float16, enabled=disp == "cuda"):
            logits = modelo(x)
        l = perda(logits.float(), y, args.perda)
        opt.zero_grad(set_to_none=True)
        escalador.scale(l).backward()
        escalador.step(opt)
        escalador.update()
        agenda.step()
        acum += l.item()
        if it % 50 == 0:
            print(f"it {it}/{args.iteracoes} perda {acum / 50:.3f} ({(time.time() - t) / 50:.2f}s/it)", flush=True)
            t, acum = time.time(), 0.0
        if it % args.validar_cada == 0 or it == args.iteracoes:
            d = validar(modelo, val, pasta)
            meta = dict(iteracao=it, dice_validacao=float(d), versao_atributos=c.VERSAO_ATRIBUTOS,
                        largura=args.largura, perda=args.perda, n_treino=len(treino))
            if d > melhor:
                melhor = d
                torch.save({"modelo": modelo.state_dict(), "meta": meta}, saida / "unet_lesoes.pt")
            torch.save({"modelo": modelo.state_dict(), "opt": opt.state_dict(), "agenda": agenda.state_dict(),
                        "iteracao": it, "melhor": melhor}, ck_path)
            print(f"== validação it {it}: Dice {d:.3f} (melhor {melhor:.3f})", flush=True)
    print(f"Fim. Melhor modelo em {saida / 'unet_lesoes.pt'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/scripts/apices.py
"""Ápices radiculares a partir dos contornos de dentes do DENTEX (conjunto de numeração).

Primeira etapa da abordagem em duas etapas: localizar as pontas das raízes para,
depois, examinar um recorte ampliado em volta de cada uma.

No DENTEX cada dente tem um polígono, o quadrante (category_id_1: 0-3 = quadrantes
FDI 1-4) e o número (category_id_2: 0-7 = dentes 1-8). O ápice é a extremidade do
polígono do lado da raiz: para cima nos quadrantes 1 e 2 (superiores) e para baixo
nos 3 e 4 (inferiores). Molares (6-8) recebem dois ápices, um em cada metade da faixa
apical, porque têm mais de uma raiz.
"""
import json
from functools import lru_cache
from pathlib import Path

import numpy as np

import comum as c

DENTEX = c.RAIZ / "Externos" / "DENTEX"
FAIXA_APICAL = 0.12  # fração da altura do dente considerada "ponta da raiz"


def _lista(v):
    return json.loads(v) if isinstance(v, str) else v


@lru_cache(maxsize=1)
def anotacoes():
    d = json.load(open(DENTEX / "train_quadrant_enumeration.json", encoding="utf-8"))
    imagens = {str(i["id"]): i for i in d["images"]}
    dentes = {}
    for a in d["annotations"]:
        pol = np.array(_lista(a["segmentation"])[0], float).reshape(-1, 2)
        dentes.setdefault(str(a["image_id"]), []).append(
            dict(poligono=pol, quadrante=int(a["category_id_1"]) + 1, dente=int(a["category_id_2"]) + 1))
    return imagens, dentes


def apices_do_dente(pol, quadrante, dente):
    """Lista de (x, y) dos ápices de um dente, em pixels da imagem original."""
    superior = quadrante in (1, 2)
    y = pol[:, 1]
    altura = y.max() - y.min()
    ext = y.min() if superior else y.max()
    faixa = pol[np.abs(y - ext) <= FAIXA_APICAL * altura]
    if dente >= 6 and len(faixa) >= 2:
        xm = faixa[:, 0].mean()
        lados = [faixa[faixa[:, 0] <= xm], faixa[faixa[:, 0] > xm]]
        pts = []
        for lado in lados:
            if len(lado):
                k = lado[:, 1].argmin() if superior else lado[:, 1].argmax()
                pts.append(tuple(lado[k]))
        if len(pts) == 2 and abs(pts[0][0] - pts[1][0]) > 0.25 * np.ptp(pol[:, 0]):
            return pts
    k = y.argmin() if superior else y.argmax()
    return [tuple(pol[k])]


def apices_da_imagem(image_id):
    """[(x, y, quadrante, dente)] de todos os ápices de uma imagem do DENTEX."""
    _, dentes = anotacoes()
    saida = []
    for t in dentes.get(str(image_id), []):
        for x, y in apices_do_dente(t["poligono"], t["quadrante"], t["dente"]):
            saida.append((float(x), float(y), t["quadrante"], t["dente"]))
    return saida


def caminho_imagem(image_id):
    imagens, _ = anotacoes()
    return DENTEX / "enumeracao" / Path(imagens[str(image_id)]["file_name"]).name

In [ ]:
%%writefile /content/scripts/duas_etapas.py
"""Abordagem em duas etapas: localizar os ápices radiculares, depois segmentar a lesão
num recorte ampliado em volta de cada ápice.

Etapa 1 (localizador): U-Net que prevê um mapa de calor de ápices na panorâmica
normalizada a 1400 px de largura (comum.LARGURA). Treinada com os contornos de dentes
do DENTEX (apices.py).

Etapa 2 (lesão): U-Net que segmenta a lesão em recortes de RECORTE x RECORTE pixels,
tirados da panorâmica normalizada a LARGURA_RECORTE (~ resolução original, o dobro da
escala de trabalho), centrados em cada ápice detectado.

Usado no Colab (treino) e localmente (avaliação/aplicação). Só depende de comum, unet,
OpenCV, NumPy e SciPy.
"""
import cv2
import numpy as np
from scipy import ndimage as ndi

import comum as c
import unet

SIGMA_APICE = 6          # px (escala 1400) do alvo gaussiano de cada ápice
LIMIAR_APICE = 0.3       # pico mínimo do mapa de calor para virar ápice
RAIO_NMS_APICE = 10      # px (escala 1400): ápices mais próximos que isso se fundem
MAX_APICES = 60          # por imagem; uma dentição completa tem ~40 ápices
LARGURA_RECORTE = 2800   # px: largura em que os recortes são tirados
RECORTE = 256            # px (escala LARGURA_RECORTE) = 128 px na escala de trabalho
MARGEM_AUMENTO = 16      # px extras guardados em volta do recorte, para deslocamento aleatório no treino


def mapa_alvo(forma, pts, sigma=SIGMA_APICE):
    """Mapa de calor alvo: máximo de gaussianas centradas nos ápices."""
    m = np.zeros(forma, np.float32)
    r = int(4 * sigma)
    for x, y in pts:
        x0, x1 = int(max(0, x - r)), int(min(forma[1], x + r + 1))
        y0, y1 = int(max(0, y - r)), int(min(forma[0], y + r + 1))
        if x0 >= x1 or y0 >= y1:
            continue
        yy, xx = np.mgrid[y0:y1, x0:x1]
        g = np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma ** 2)).astype(np.float32)
        m[y0:y1, x0:x1] = np.maximum(m[y0:y1, x0:x1], g)
    return m


def picos(p, limiar=LIMIAR_APICE, raio=RAIO_NMS_APICE, maximo=MAX_APICES):
    """Coordenadas (x, y) dos máximos locais do mapa de calor, no máximo `maximo`, os mais fortes primeiro."""
    ps = cv2.GaussianBlur(p, (0, 0), 1.5)
    m = (ps == ndi.maximum_filter(ps, size=2 * raio + 1)) & (ps >= limiar)
    ys, xs = np.nonzero(m)
    ordem = np.argsort(-ps[ys, xs])[:maximo]
    return np.column_stack([xs[ordem], ys[ordem]]).astype(np.float32)


def detectar_apices(localizador, img_trabalho):
    """Ápices (x, y) na escala de trabalho (imagem normalizada a comum.LARGURA)."""
    return picos(unet.prever(localizador, img_trabalho))


def recorte_em(img, cx, cy, tam, preencher=0):
    """Recorte tam x tam centrado em (cx, cy), completando fora da imagem com `preencher`."""
    h, w = img.shape[:2]
    x0, y0 = int(round(cx)) - tam // 2, int(round(cy)) - tam // 2
    out = np.full((tam, tam), preencher, img.dtype)
    sx0, sy0 = max(0, x0), max(0, y0)
    sx1, sy1 = min(w, x0 + tam), min(h, y0 + tam)
    if sx1 > sx0 and sy1 > sy0:
        out[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = img[sy0:sy1, sx0:sx1]
    return out, (x0, y0)


def probabilidade_duas_etapas(localizador, segmentador, original):
    """Mapa de probabilidade na escala de trabalho a partir das duas etapas.

    Fora dos recortes a probabilidade é 0: a etapa 2 só examina a vizinhança dos
    ápices detectados. Recortes sobrepostos são combinados pelo máximo.
    Devolve (prob_trabalho, apices_trabalho).
    """
    img_t, esc_t = c.normalizar(original)
    apices = detectar_apices(localizador, img_t)
    img_r, esc_r = c.normalizar(original, LARGURA_RECORTE)
    fator = esc_r / esc_t
    prob_r = np.zeros(img_r.shape, np.float32)
    for x, y in apices:
        rec, (x0, y0) = recorte_em(img_r, x * fator, y * fator, RECORTE)
        p = unet.prever(segmentador, rec)
        sx0, sy0 = max(0, x0), max(0, y0)
        sx1, sy1 = min(img_r.shape[1], x0 + RECORTE), min(img_r.shape[0], y0 + RECORTE)
        if sx1 > sx0 and sy1 > sy0:
            alvo = prob_r[sy0:sy1, sx0:sx1]
            np.maximum(alvo, p[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0], out=alvo)
    prob_t = cv2.resize(prob_r, (img_t.shape[1], img_t.shape[0]), interpolation=cv2.INTER_AREA)
    return prob_t, apices

In [ ]:
%%writefile /content/scripts/treino_duas_etapas.py
"""Treina as duas etapas no Colab (GPU): localizador de ápices (DENTEX) e U-Net de
lesão em recortes ampliados em volta dos ápices (Hanói).

Fases, cada uma pulada se o resultado já estiver em --saida (permite retomar):
  1. localizador_apices.pt   U-Net de mapa de calor treinada nos ápices do DENTEX
  2. apices_hanoi.json       ápices detectados pelo localizador nas panorâmicas de Hanói
  3. recortes/               recortes (imagem + rótulo) em volta desses ápices
  4. lesoes_recortes.pt      U-Net de lesão treinada nos recortes

Uso: python treino_duas_etapas.py --saida PASTA [--its1 3000] [--its2 6000]
     [--perda tversky] [--limite-dentex N --limite-hanoi N --sem-pre-treino]  (testes)
"""
import argparse
import json
import math
import os
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from joblib import Parallel, delayed
from torch.utils.data import DataLoader, Dataset

import apices as ap
import comum as c
import duas_etapas as de
from treino_unet import IGNORAR, perda
from unet import UNetResNet34, prever

TOTAL = de.RECORTE + 2 * de.MARGEM_AUMENTO
N_NEGATIVOS = 4      # recortes sem lesão guardados por imagem
MAX_POSITIVOS = 8    # recortes com lesão guardados por imagem (os ápices mais próximos da lesão)
MIN_LESAO_PX = 20    # pixels de lesão no recorte central para contar como positivo


def log(msg):
    print(msg, flush=True)


def agenda_cos(opt, total, aquecimento=200):
    return torch.optim.lr_scheduler.LambdaLR(
        opt, lambda it: min(1, (it + 1) / aquecimento) * 0.5 * (1 + math.cos(math.pi * min(it, total) / total)))


# ------------------------------------------------------------------ fase 1: localizador
class RecortesApices(Dataset):
    def __init__(self, dados, n, semente=0, tam=(512, 768)):
        self.dados, self.n, self.semente, self.tam = dados, n, semente, tam

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        rng = np.random.default_rng(self.semente * 1_000_003 + i)
        img, pts = self.dados[rng.integers(len(self.dados))]
        ch, cw = self.tam
        h, w = img.shape
        y0, x0 = rng.integers(0, h - ch + 1), rng.integers(0, w - cw + 1)
        a = img[y0:y0 + ch, x0:x0 + cw].astype(np.float32) / 255
        p = pts - np.array([x0, y0], np.float32)
        if rng.random() < .5:
            a = a[:, ::-1]
            p = p.copy(); p[:, 0] = cw - 1 - p[:, 0]
        alvo = de.mapa_alvo((ch, cw), p)
        a = np.clip(a ** rng.uniform(0.7, 1.4) * rng.uniform(0.85, 1.15) + rng.uniform(-0.08, 0.08), 0, 1)
        return torch.from_numpy(a.copy())[None], torch.from_numpy(alvo)[None]


def avaliar_localizador(modelo, val):
    modelo.eval()
    vp = fn = fp = 0
    for img, pts in val:
        det = de.picos(prever(modelo, img, espelhar=False))
        if len(det) and len(pts):
            d = np.sqrt(((pts[:, None] - det[None]) ** 2).sum(-1))
            vp_i = int((d.min(1) <= 20).sum())
            fp += int((d.min(0) > 20).sum())
        else:
            vp_i = 0
            fp += len(det)
        vp += vp_i; fn += len(pts) - vp_i
    modelo.train()
    rec, prec = vp / max(vp + fn, 1), vp / max(vp + fp, 1)
    return rec, prec, 2 * rec * prec / max(rec + prec, 1e-9)


def fase_localizador(args, disp, saida):
    arq = saida / "localizador_apices.pt"
    if arq.exists():
        log("Fase 1: localizador já treinado, carregando")
        return
    imagens, _ = ap.anotacoes()
    ids = sorted(imagens, key=int)
    np.random.default_rng(0).shuffle(ids)
    if args.limite_dentex:
        ids = ids[:args.limite_dentex]
    n_val = min(50, len(ids) // 5)

    def carregar(i):
        img, esc = c.normalizar(c.ler_cinza(ap.caminho_imagem(i)))
        pts = np.array([(x * esc, y * esc) for x, y, _, _ in ap.apices_da_imagem(i)], np.float32).reshape(-1, 2)
        return img, pts

    t = time.time()
    dados = Parallel(n_jobs=max(1, args.workers))(delayed(carregar)(i) for i in ids)
    val, tr = dados[:n_val], dados[n_val:]
    log(f"Fase 1: DENTEX {len(tr)} treino / {len(val)} validação ({time.time() - t:.0f}s)")

    modelo = UNetResNet34(pre_treinada=not args.sem_pre_treino).to(disp)
    opt = torch.optim.AdamW(modelo.parameters(), lr=3e-4, weight_decay=1e-4)
    agenda = agenda_cos(opt, args.its1)
    escal = torch.amp.GradScaler(enabled=disp == "cuda")
    carga = DataLoader(RecortesApices(tr, args.its1 * args.lote1), batch_size=args.lote1,
                       num_workers=args.workers, pin_memory=disp == "cuda")
    melhor, t, acum = -1.0, time.time(), 0.0
    modelo.train()
    for it, (x, y) in enumerate(carga, 1):
        x, y = x.to(disp), y.to(disp)
        with torch.autocast(device_type=disp, dtype=torch.float16, enabled=disp == "cuda"):
            logit = modelo(x)
        l = (F.binary_cross_entropy_with_logits(logit.float(), y, reduction="none") * (1 + 20 * y)).mean()
        opt.zero_grad(set_to_none=True); escal.scale(l).backward(); escal.step(opt); escal.update(); agenda.step()
        acum += l.item()
        if it % 50 == 0:
            log(f"  [1] it {it}/{args.its1} perda {acum / 50:.4f} ({(time.time() - t) / 50:.2f}s/it)")
            t, acum = time.time(), 0.0
        if it % args.validar_cada == 0 or it == args.its1:
            rec, prec, f1 = avaliar_localizador(modelo, val)
            if f1 > melhor:
                melhor = f1
                torch.save({"modelo": modelo.state_dict(),
                            "meta": dict(iteracao=it, recall_20px=rec, precisao_20px=prec, f1=f1,
                                         sigma=de.SIGMA_APICE, largura=c.LARGURA)}, arq)
            log(f"  == [1] validação it {it}: ápices a ≤20 px recall {rec:.3f} precisão {prec:.3f} F1 {f1:.3f} (melhor {melhor:.3f})")


# ------------------------------------------------------------------ fase 2: ápices em Hanói
class ImagensHanoi(Dataset):
    def __init__(self, nomes):
        self.nomes = nomes

    def __len__(self):
        return len(self.nomes)

    def __getitem__(self, i):
        img, _ = c.normalizar(c.ler_cinza(c.HANOI / "imagens" / "Original JPG Images" / f"{self.nomes[i]}.jpg"))
        return self.nomes[i], img


def fase_apices_hanoi(args, disp, saida, nomes):
    arq = saida / "apices_hanoi.json"
    if arq.exists():
        log("Fase 2: ápices de Hanói já detectados")
        return json.load(open(arq))
    ck = torch.load(saida / "localizador_apices.pt", map_location=disp, weights_only=False)
    modelo = UNetResNet34().to(disp)
    modelo.load_state_dict(ck["modelo"]); modelo.eval()
    res, t = {}, time.time()
    carga = DataLoader(ImagensHanoi(nomes), batch_size=None, num_workers=args.workers)
    for k, (nm, img) in enumerate(carga, 1):
        res[nm] = de.detectar_apices(modelo, np.asarray(img)).round(1).tolist()
        if k % 250 == 0:
            log(f"  [2] {k}/{len(nomes)} ({(time.time() - t) / k:.2f}s/imagem)")
    json.dump(res, open(arq, "w"))
    n = [len(v) for v in res.values()]
    log(f"Fase 2: {sum(n)} ápices em {len(n)} imagens (mediana {np.median(n):.0f} por imagem)")
    return res


# ------------------------------------------------------------------ fase 3: recortes
def recortes_da_imagem(nome, pts, semente):
    """Recortes TOTAL x TOTAL (imagem e rótulo 0/1/2) em volta dos ápices de uma imagem de Hanói."""
    img, m, inc = c.imagem_hanoi(nome, de.LARGURA_RECORTE)
    rot = np.zeros(img.shape, np.uint8)
    rot[inc] = IGNORAR
    rot[m] = 1
    fator = de.LARGURA_RECORTE / c.LARGURA
    pos, neg = [], []
    for x, y in pts:
        ri, _ = de.recorte_em(img, x * fator, y * fator, TOTAL, 0)
        rr, _ = de.recorte_em(rot, x * fator, y * fator, TOTAL, IGNORAR)
        centro = rr[de.MARGEM_AUMENTO:-de.MARGEM_AUMENTO, de.MARGEM_AUMENTO:-de.MARGEM_AUMENTO]
        n_les = int((centro == 1).sum())
        (pos if n_les >= MIN_LESAO_PX else neg).append((n_les, ri, rr))
    rng = np.random.default_rng(semente)
    pos = sorted(pos, key=lambda t: -t[0])[:MAX_POSITIVOS]
    neg = [neg[k] for k in rng.permutation(len(neg))[:N_NEGATIVOS]]
    sel = pos + neg
    if not sel:
        return None
    return (np.stack([s[1] for s in sel]), np.stack([s[2] for s in sel]),
            np.array([1] * len(pos) + [0] * len(neg), np.uint8))


def fase_recortes(args, saida, apices_hanoi, nomes):
    pasta = Path(args.pasta_recortes)
    pasta.mkdir(parents=True, exist_ok=True)
    pronto = pasta / "pronto.json"
    if pronto.exists():
        log("Fase 3: recortes já preparados")
        return json.load(open(pronto))
    t, partes, bloco = time.time(), [], 200
    for b in range(0, len(nomes), bloco):
        lote = nomes[b:b + bloco]
        res = Parallel(n_jobs=max(1, args.workers))(delayed(recortes_da_imagem)(nm, apices_hanoi.get(nm, []), k)
                                            for k, nm in enumerate(lote, b))
        res = [(nm, r) for nm, r in zip(lote, res) if r is not None]
        if res:
            np.save(pasta / f"img_{b:05d}.npy", np.concatenate([r[0] for _, r in res]))
            np.save(pasta / f"rot_{b:05d}.npy", np.concatenate([r[1] for _, r in res]))
            np.save(pasta / f"pos_{b:05d}.npy", np.concatenate([r[2] for _, r in res]))
            partes.append(dict(parte=f"{b:05d}", nomes=[nm for nm, r in res for _ in range(len(r[2]))]))
        log(f"  [3] {min(b + bloco, len(nomes))}/{len(nomes)} imagens ({time.time() - t:.0f}s)")
    json.dump(partes, open(pronto, "w"))
    return partes


class Recortes(Dataset):
    def __init__(self, pasta, partes, nomes_ok, n, semente=0):
        self.img, self.rot, idx_pos, idx_neg = [], [], [], []
        for k, p in enumerate(partes):
            self.img.append(np.load(Path(pasta) / f"img_{p['parte']}.npy", mmap_mode="r"))
            self.rot.append(np.load(Path(pasta) / f"rot_{p['parte']}.npy", mmap_mode="r"))
            pos = np.load(Path(pasta) / f"pos_{p['parte']}.npy")
            for j, nm in enumerate(p["nomes"]):
                if nm in nomes_ok:
                    (idx_pos if pos[j] else idx_neg).append((k, j))
        self.pos, self.neg, self.n, self.semente = idx_pos, idx_neg, n, semente

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        rng = np.random.default_rng(self.semente * 1_000_003 + i)
        fonte = self.pos if (rng.random() < 0.5 and self.pos) or not self.neg else self.neg
        k, j = fonte[rng.integers(len(fonte))]
        dy, dx = rng.integers(0, 2 * de.MARGEM_AUMENTO + 1, size=2)
        a = self.img[k][j, dy:dy + de.RECORTE, dx:dx + de.RECORTE].astype(np.float32) / 255
        r = np.array(self.rot[k][j, dy:dy + de.RECORTE, dx:dx + de.RECORTE])
        if rng.random() < .5:
            a, r = a[:, ::-1], r[:, ::-1]
        a = np.clip(a ** rng.uniform(0.7, 1.4) * rng.uniform(0.85, 1.15) + rng.uniform(-0.08, 0.08), 0, 1)
        return torch.from_numpy(a.copy())[None], torch.from_numpy(r.copy()).long()


def avaliar_segmentador(modelo, ds, disp, lote=64):
    """Dice nos recortes de validação (positivos e negativos, sem deslocamento)."""
    modelo.eval()
    inter = soma = 0
    m = de.MARGEM_AUMENTO
    with torch.no_grad():
        for fonte in (ds.pos, ds.neg):
            for b in range(0, len(fonte), lote):
                xs, rs = [], []
                for k, j in fonte[b:b + lote]:
                    xs.append(ds.img[k][j, m:m + de.RECORTE, m:m + de.RECORTE].astype(np.float32) / 255)
                    rs.append(np.array(ds.rot[k][j, m:m + de.RECORTE, m:m + de.RECORTE]))
                x = torch.from_numpy(np.stack(xs))[:, None].to(disp)
                p = (torch.sigmoid(modelo(x))[:, 0] >= 0.5).cpu().numpy()
                r = np.stack(rs)
                inter += (p & (r == 1)).sum()
                soma += (p & (r != IGNORAR)).sum() + (r == 1).sum()
    modelo.train()
    return 2 * inter / max(soma, 1)


def fase_segmentador(args, disp, saida, partes, nomes):
    arq = saida / "lesoes_recortes.pt"
    if arq.exists():
        log("Fase 4: segmentador já treinado")
        return
    val_nomes = set(nomes[:max(1, min(200, len(nomes) // 5))])
    tr_nomes = set(nomes) - val_nomes
    ds_tr = Recortes(args.pasta_recortes, partes, tr_nomes, args.its2 * args.lote2)
    ds_val = Recortes(args.pasta_recortes, partes, val_nomes, 1)
    log(f"Fase 4: recortes de treino {len(ds_tr.pos)} com lesão / {len(ds_tr.neg)} sem; "
        f"validação {len(ds_val.pos)} / {len(ds_val.neg)}")
    modelo = UNetResNet34(pre_treinada=not args.sem_pre_treino).to(disp)
    opt = torch.optim.AdamW(modelo.parameters(), lr=3e-4, weight_decay=1e-4)
    agenda = agenda_cos(opt, args.its2)
    escal = torch.amp.GradScaler(enabled=disp == "cuda")
    carga = DataLoader(ds_tr, batch_size=args.lote2, num_workers=args.workers, pin_memory=disp == "cuda")
    melhor, t, acum = -1.0, time.time(), 0.0
    modelo.train()
    for it, (x, y) in enumerate(carga, 1):
        x, y = x.to(disp), y.to(disp)
        with torch.autocast(device_type=disp, dtype=torch.float16, enabled=disp == "cuda"):
            logit = modelo(x)
        l = perda(logit.float(), y, args.perda)
        opt.zero_grad(set_to_none=True); escal.scale(l).backward(); escal.step(opt); escal.update(); agenda.step()
        acum += l.item()
        if it % 50 == 0:
            log(f"  [4] it {it}/{args.its2} perda {acum / 50:.3f} ({(time.time() - t) / 50:.2f}s/it)")
            t, acum = time.time(), 0.0
        if it % args.validar_cada == 0 or it == args.its2:
            d = avaliar_segmentador(modelo, ds_val, disp)
            if d > melhor:
                melhor = d
                torch.save({"modelo": modelo.state_dict(),
                            "meta": dict(iteracao=it, dice_validacao=float(d), perda=args.perda,
                                         recorte=de.RECORTE, largura_recorte=de.LARGURA_RECORTE)}, arq)
            log(f"  == [4] validação it {it}: Dice {d:.3f} (melhor {melhor:.3f})")


def main():
    a = argparse.ArgumentParser()
    a.add_argument("--saida", required=True)
    a.add_argument("--pasta-recortes", default=str(c.HANOI / "recortes_apices"))
    a.add_argument("--its1", type=int, default=3000)
    a.add_argument("--its2", type=int, default=6000)
    a.add_argument("--lote1", type=int, default=8)
    a.add_argument("--lote2", type=int, default=32)
    a.add_argument("--validar-cada", type=int, default=500)
    a.add_argument("--perda", choices=("dice", "tversky"), default="tversky")
    a.add_argument("--workers", type=int, default=min(4, os.cpu_count() or 2))
    a.add_argument("--limite-dentex", type=int, default=0)
    a.add_argument("--limite-hanoi", type=int, default=0)
    a.add_argument("--sem-pre-treino", action="store_true")
    args = a.parse_args()
    disp = "cuda" if torch.cuda.is_available() else "cpu"
    saida = Path(args.saida)
    saida.mkdir(parents=True, exist_ok=True)
    log(f"Dispositivo: {disp} | CPUs: {os.cpu_count()} | workers: {args.workers}")

    fase_localizador(args, disp, saida)
    nomes = c.nomes_hanoi()
    np.random.default_rng(1).shuffle(nomes)
    if args.limite_hanoi:
        nomes = nomes[:args.limite_hanoi]
    apices_hanoi = fase_apices_hanoi(args, disp, saida, nomes)
    partes = fase_recortes(args, saida, apices_hanoi, nomes)
    fase_segmentador(args, disp, saida, partes, nomes)
    log(f"Fim. Modelos em {saida}: localizador_apices.pt e lesoes_recortes.pt")


if __name__ == "__main__":
    main()

### 4. Baixar e extrair o conjunto de Hanói (~850 MB)

In [ ]:
%%bash
set -e
mkdir -p /content/Externos/Hanoi/imagens /content/Externos/Hanoi/anotacoes
cd /content/Externos/Hanoi
if [ ! -d "imagens/Original JPG Images" ]; then
  wget -q -O originais.rar "https://data.mendeley.com/public-files/datasets/kx52tk2ddj/files/0d016bd4-d5ac-4363-ac35-c74d87ba4022/file_downloaded"
  wget -q -O annots.rar "https://data.mendeley.com/public-files/datasets/kx52tk2ddj/files/fa6e54cc-4094-4165-9987-3a8589213696/file_downloaded"
  which unrar > /dev/null || (apt-get -qq update && apt-get -qq install -y unrar > /dev/null)
  unrar x -o+ -inul originais.rar imagens/
  unrar x -o+ -inul annots.rar anotacoes/
  rm originais.rar annots.rar
fi
echo "imagens: $(ls 'imagens/Original JPG Images' | wc -l)  anotações: $(ls 'anotacoes/Image Annots' | wc -l)"

### 5. Baixar só o conjunto de numeração de dentes do DENTEX (~1,7 GB, lido de dentro do zip remoto)

In [ ]:
!pip -q install fsspec aiohttp
import os, zipfile, time, fsspec
D = '/content/Externos/DENTEX'
os.makedirs(D + '/enumeracao', exist_ok=True)
z = zipfile.ZipFile(fsspec.open('https://huggingface.co/datasets/ibrahimhamamci/DENTEX/resolve/main/DENTEX/training_data.zip', block_size=2**23).open())
pre = 'training_data/quadrant_enumeration/'
js = D + '/train_quadrant_enumeration.json'
if not os.path.exists(js):
    open(js, 'wb').write(z.read(pre + 'train_quadrant_enumeration.json'))
nomes = [n for n in z.namelist() if n.startswith(pre + 'xrays/') and n.endswith('.png')]
t = time.time()
for k, n in enumerate(nomes, 1):
    dst = D + '/enumeracao/' + n.split('/')[-1]
    if not os.path.exists(dst):
        open(dst, 'wb').write(z.read(n))
    if k % 100 == 0:
        print(k, len(nomes), f'{time.time() - t:.0f}s', flush=True)
print('DENTEX:', len(os.listdir(D + '/enumeracao')), 'imagens')

### 6. Treinar as duas etapas (fases já prontas no Drive são puladas)

In [ ]:
!cd /content/scripts && python -u treino_duas_etapas.py --saida "{SAIDA}" --its1 3000 --its2 6000

### 7. Baixar os modelos (salve em `ML Odonto/Modelos/duas_etapas/`)

In [ ]:
from google.colab import files
for nome in ('localizador_apices.pt', 'lesoes_recortes.pt', 'apices_hanoi.json'):
    files.download(f'{SAIDA}/{nome}')